# Gold — dim_ticker (SCD Type 2)

`silver.ticker` → **`gold.dim_ticker`**.

One row per **version** of a ticker: the **99 UK trusts that are still listed**, plus SPY, IVV
and VOO. That includes the 7 whose history cannot be trusted and the 3 with no usable prices,
so the dimension stays honest about which trusts the results cannot cover and why.

A new version opens when **`manager` or `management_group`** changes. Nothing else versions:
`aic_sector` never changes, and the coverage counts change every month, which would version
every trust monthly — noise dressed as history.

**Why `status` is not a driver.** The dimension is defined as the trusts that are *currently
listed*, so it only ever holds `status = 'active'`. A status that cannot change cannot drive a
version. A trust that winds up does not get a closed version here — it leaves the table, by
the third statement below. That is a hard delete from a type-2 dimension, which is unusual and
deliberate: see `specs/08_listed_only/listed-only.md`.

The manager file is a **snapshot with no dates**, so the first run makes everything version 1
and history accrues from the second run onward. Proving it works is the demo: change one
manager, re-run, and that ticker has two rows.

Expected: **102 rows, all current**.

In [0]:
CREATE OR REPLACE TEMP VIEW gold_stage_ticker AS
WITH clock AS (
  -- The last complete month. A change detected now takes effect from the month after it,
  -- so a new version never overlaps the one it replaces.
  SELECT MAX(month_key)                                                    AS latest_month,
         CAST(DATE_FORMAT(ADD_MONTHS(MAX(month_start), 1), 'yyyyMM') AS INT) AS next_month,
         CAST(DATE_FORMAT(MIN(month_start), 'yyyyMM') AS INT)              AS earliest_month
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
),
labelled AS (
  -- Absence is labelled, never left blank or null, and the two kinds are kept apart:
  -- an index has no manager because it is an index, which is not missing information.
  SELECT s.ticker, s.trust_name, s.entity_type,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.aic_sector), ''), 'NoInfo') END        AS aic_sector,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.manager), ''), 'NoInfo') END           AS manager,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.management_group), ''), 'NoInfo') END  AS management_group,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.manager_structure), ''), 'NoInfo') END AS manager_structure,
         s.status,
         CASE WHEN s.entity_type = 'Index' THEN s.currency
              ELSE COALESCE(NULLIF(TRIM(s.currency), ''), 'NoInfo') END          AS currency,
         s.price_source, s.data_status,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.source_url), ''), 'NoInfo') END        AS source_url,
         s.first_month, s.last_month, s.months_available
  FROM `index-vs-trust-pipeline`.silver.ticker s
),
dated AS (
  SELECT l.*,
         c.latest_month,
         -- A ticker we have never seen starts at its own first month: the snapshot is
         -- assumed to have held from the beginning, and the spec says so openly. Any later
         -- version starts in the month after the change was detected.
         CASE WHEN EXISTS (SELECT 1 FROM `index-vs-trust-pipeline`.gold.dim_ticker d
                           WHERE d.ticker = l.ticker)
              THEN c.next_month
              ELSE COALESCE(l.first_month, c.earliest_month)
         END AS effective_start_month
  FROM labelled l CROSS JOIN clock c
)
SELECT MD5(CONCAT_WS('|', ticker, CAST(effective_start_month AS STRING))) AS ticker_key,
       ticker, trust_name, entity_type, aic_sector, manager, management_group,
       manager_structure, status, currency, price_source, data_status, source_url,
       first_month, last_month, months_available,
       effective_start_month,
       CAST(NULL AS INT) AS effective_end_month,
       true              AS is_current,
       latest_month
FROM dated;

## The SCD2, in three statements

**Close the old version, open the new one, then remove whoever has left the universe.** Three
plain statements rather than the usual single MERGE with a null merge key — the clever version
is one statement, this version is three sentences.

The second statement inserts for a brand-new ticker **and** for one whose current version the
first statement just closed, because in both cases no current row is left to match.

The third is the one that needs defending. A type-2 dimension normally never deletes: that is
the whole point of keeping history. This one does, because the table is defined as *the trusts
currently listed*, not *every trust that ever listed*. A wound-up trust is not a new version of
itself, it is outside the universe the project reports on. The record that it existed lives in
`landing.yf_pull_log_raw`, which keeps the source's own refusal message against every symbol
that returned nothing.

In [ ]:
-- 1. Close any version whose driver attributes have changed.
--    Null-safe (<=>) so a missing manager is never mistaken for a change.
--    status is not a driver: the table only ever holds active rows, so it cannot change here.
MERGE INTO `index-vs-trust-pipeline`.gold.dim_ticker AS t
USING gold_stage_ticker AS s
   ON t.ticker = s.ticker AND t.is_current
WHEN MATCHED AND NOT (t.manager          <=> s.manager
                  AND t.management_group <=> s.management_group)
THEN UPDATE SET t.is_current          = false,
                t.effective_end_month = s.latest_month;

In [0]:
-- 2. Open a version for anything with no current row: new tickers, and the ones just closed.
MERGE INTO `index-vs-trust-pipeline`.gold.dim_ticker AS t
USING gold_stage_ticker AS s
   ON t.ticker = s.ticker AND t.is_current
WHEN NOT MATCHED THEN INSERT (
  ticker_key, ticker, trust_name, entity_type, aic_sector, manager, management_group,
  manager_structure, status, currency, price_source, data_status, source_url,
  first_month, last_month, months_available,
  effective_start_month, effective_end_month, is_current
) VALUES (
  s.ticker_key, s.ticker, s.trust_name, s.entity_type, s.aic_sector, s.manager,
  s.management_group, s.manager_structure, s.status, s.currency, s.price_source,
  s.data_status, s.source_url, s.first_month, s.last_month, s.months_available,
  s.effective_start_month, s.effective_end_month, s.is_current
);

In [ ]:
-- 3. Remove every version of a ticker that has left the universe. Matched on ticker alone,
--    not on is_current, so a closed version goes with its current one and no orphan key is
--    left behind for the facts to point at.
--    A hard delete from a type-2 dimension, on purpose: the table is the trusts currently
--    listed, and the record that a wound-up trust existed lives in landing.yf_pull_log_raw.
MERGE INTO `index-vs-trust-pipeline`.gold.dim_ticker AS t
USING (SELECT DISTINCT ticker FROM gold_stage_ticker) AS s
   ON t.ticker = s.ticker
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Verification

In [ ]:
SELECT COUNT(*)                                                      AS rows_total,
       SUM(CASE WHEN is_current THEN 1 ELSE 0 END)                   AS current_rows,
       SUM(CASE WHEN effective_end_month IS NOT NULL THEN 1 ELSE 0 END) AS closed_rows,
       COUNT(*) - COUNT(DISTINCT ticker_key)                         AS duplicate_keys,
       COUNT(DISTINCT ticker)                                        AS distinct_tickers,
       SUM(CASE WHEN entity_type = 'Index'     THEN 1 ELSE 0 END)    AS index_rows,
       SUM(CASE WHEN status <> 'active'        THEN 1 ELSE 0 END)    AS not_listed,
       SUM(CASE WHEN data_status = 'excluded'  THEN 1 ELSE 0 END)    AS excluded
FROM `index-vs-trust-pipeline`.gold.dim_ticker;

Expect **102 / 102 / 0 / 0 / 102 / 3 / 0 / 7**.

- **`not_listed` must be 0.** The dimension holds the currently listed universe, and the third
  statement is what keeps it that way. On the run that applies this change it removes **19**
  rows that were already in the table — a filter on the staging view alone would have left
  every one of them behind, because neither of the first two statements can remove a row.
- `rows_total`, `current_rows` and `distinct_tickers` are all **102** only while every ticker
  is on version 1. After the SCD2 demo, `rows_total` rises and `distinct_tickers` stays at
  102 — that gap *is* the history.
- **excluded 7** — `CLDN`, `JEMA`, `MRC`, `MYI`, `NAS`, `PCT`, `WWH`. All still listed, all
  still named here, none of them scoreable.

In [0]:
-- Nothing is blank, and the two kinds of absence stay apart.
SELECT SUM(CASE WHEN manager = 'NoInfo'                THEN 1 ELSE 0 END) AS manager_noinfo,
       SUM(CASE WHEN management_group = 'NoInfo'       THEN 1 ELSE 0 END) AS group_noinfo,
       SUM(CASE WHEN manager = 'NotApplicable'         THEN 1 ELSE 0 END) AS manager_na,
       SUM(CASE WHEN manager_structure = 'multi'       THEN 1 ELSE 0 END) AS multi,
       SUM(CASE WHEN manager_structure = 'sole'        THEN 1 ELSE 0 END) AS sole,
       COUNT(DISTINCT management_group)                                   AS distinct_groups,
       SUM(CASE WHEN manager IS NULL OR management_group IS NULL
                  OR manager_structure IS NULL THEN 1 ELSE 0 END)         AS any_nulls_left,
       SUM(CASE WHEN source_url LIKE 'http%'           THEN 1 ELSE 0 END) AS real_source_urls
FROM `index-vs-trust-pipeline`.gold.dim_ticker;

Expect **2 / 2 / 3 / 70 / 27 / 53 / 0 / 99**.

- `any_nulls_left` must be **0** — every row carries a readable label.
- `manager_noinfo` is **2**, down from 21. Every one of the 19 trusts that left was wound up,
  so none of them had a current manager in the metadata file; what is left is 2 listed trusts
  whose manager the file leaves blank.
- `distinct_groups` is **53**: **51** real management groups, plus `NoInfo` and
  `NotApplicable`. One group disappeared with the delisted trusts — it had no listed trust
  left. The two labels are deliberate, so the dashboard can show how many trusts publish no
  group rather than hiding them in a null.
- `real_source_urls` is **99** — every listed trust cites where its metadata came from.

In [ ]:
-- Every trust in the dimension is listed; data_status is what separates the ones the results
-- can cover from the ones they cannot. Both live in the dimension, so "scoreable or not" is
-- one fact table and a WHERE clause, never a second table.
SELECT status, data_status, COUNT(*) AS tickers
FROM `index-vs-trust-pipeline`.gold.dim_ticker
WHERE is_current
GROUP BY status, data_status
ORDER BY status, data_status;

Expect **one value of `status` — `active` — across four `data_status` rows**:

| data_status | tickers |
|---|---|
| `excluded` | **7** |
| `no-data` | **2** |
| `stub` | **1** |
| `usable` | **92** |

92 usable includes the 3 index tickers, so **89 trusts** can be scored, and **10** listed
trusts cannot. The dimension names all 10 rather than letting them go quietly missing from a
count — which is the question this table answers: *"which trusts are not in your results, and
why?"*

A second value of `status` appearing here would mean the third statement did not run.

In [ ]:
-- Four rows that each stress a different part of the design.
SELECT ticker, entity_type, manager, management_group, manager_structure,
       status, data_status, price_source, months_available,
       effective_start_month, effective_end_month, is_current
FROM `index-vs-trust-pipeline`.gold.dim_ticker
WHERE ticker IN ('SMT', 'SPY', 'CLDN', 'BSIF')
ORDER BY CASE ticker WHEN 'SMT' THEN 1 WHEN 'SPY' THEN 2 WHEN 'CLDN' THEN 3 ELSE 4 END;

Expect:

| ticker | why it is here |
|---|---|
| `SMT` | the ordinary case — Baillie Gifford, `multi`, 177 months, scoreable |
| `SPY` | why `NotApplicable` exists: an index has no manager **because it is an index** |
| `CLDN` | **listed, named, and not in the results** — Yahoo offers a full history, but its dividends reach 642% of its share price, so `data_status = 'excluded'` and `months_available = 0` |
| `BSIF` | a `stub`: one usable month, below the 36-month floor Gold applies |

`CLDN` is the row that matters now. It is the case a pipeline is most tempted to hide — a
trust it could not process — and it is here, listed, with the reason attached, rather than
silently absent from a denominator.

Every row in this table reads `active`, because the dimension is the listed universe. A trust
that winds up is removed by the third statement, and the evidence that it existed stays in
`landing.yf_pull_log_raw`.